# 西部量化可转债策略 V3.0 - 量化研究平台

本Notebook提供了策略研究和分析的示例代码。

## 目录
1. 数据获取与预处理
2. 因子研究与可视化
3. 回测分析
4. 策略优化
5. 风险分析

In [ ]:
# 导入依赖
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import date, timedelta

# 设置绘图风格
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.sans-serif'] = ['Arial Unicode MS', 'SimHei']
plt.rcParams['axes.unicode_minus'] = False

print("环境初始化完成")

## 1. 数据获取与预处理

In [ ]:
# 导入策略模块（离线/打包环境下可能不可用）
class StopExecution(Exception):
    """在 Notebook cell 中 raise 可跳过后续执行，替代 if/else 缩进"""
    pass

# 注册 StopExecution 拦截器，避免 Jupyter 显示红色 traceback
# 兼容 IPython < 8 (custom_exceptions) 和 Jupyter 7+ (try/except)
try:
    _ip = get_ipython()
    if hasattr(_ip, "custom_exceptions"):
        # IPython < 8 / Jupyter Notebook 经典版
        if StopExecution not in (_ip.custom_exceptions or ()):
            _ip.custom_exceptions = (_ip.custom_exceptions or ()) + (StopExecution,)
    else:
        # Jupyter 7+ / JupyterLab 4+: 降级为 try/except 模式
        pass  # StopExecution 由 cell 内 try/except 捕获
except NameError:
    pass  # 非 IPython 环境

try:
    from app.xb_strategy.core.data_adapter import AkshareDataSource
    from app.xb_strategy.core.scoring import SevenDimScoringEngine
    from app.xb_strategy.core.filters import VetoFilter
    from app.xb_strategy.core.backtest import BacktestEngine, BacktestConfig
    DATA_MODULES_AVAILABLE = True
    print("策略模块导入成功")
except ImportError as e:
    DATA_MODULES_AVAILABLE = False
    print(f"⚠️ 策略模块导入失败（离线/Electron打包环境）: {e}")
    print("将使用模拟数据进行后续分析")


## 2. 因子研究与可视化

In [ ]:
# 因子分布分析
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 转股溢价率分布
axes[0, 0].hist(cb_data['conversion_premium'] * 100, bins=50, edgecolor='white', alpha=0.7)
axes[0, 0].set_title('转股溢价率分布', fontsize=12)
axes[0, 0].set_xlabel('溢价率 (%)')
axes[0, 0].set_ylabel('频数')

# YTM分布
axes[0, 1].hist(cb_data['ytm'] * 100, bins=50, edgecolor='white', alpha=0.7, color='green')
axes[0, 1].set_title('到期收益率分布', fontsize=12)
axes[0, 1].set_xlabel('YTM (%)')
axes[0, 1].set_ylabel('频数')

# 剩余年限分布
axes[1, 0].hist(cb_data['remaining_years'], bins=30, edgecolor='white', alpha=0.7, color='orange')
axes[1, 0].set_title('剩余年限分布', fontsize=12)
axes[1, 0].set_xlabel('年限 (年)')
axes[1, 0].set_ylabel('频数')

# 换手率分布
axes[1, 1].hist(cb_data['turnover_rate'], bins=50, edgecolor='white', alpha=0.7, color='purple')
axes[1, 1].set_title('换手率分布', fontsize=12)
axes[1, 1].set_xlabel('换手率 (%)')
axes[1, 1].set_ylabel('频数')

plt.tight_layout()
plt.show()

In [ ]:
# 因子相关性分析
factor_cols = ['conversion_premium', 'ytm', 'remaining_years', 'turnover_rate', 'stock_change_pct']
corr_matrix = cb_data[factor_cols].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, cmap='RdYlGn', center=0, fmt='.2f')
plt.title('因子相关性矩阵', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Jupyter 7+ 兼容：StopExecution 由 try/except 捕获
try:
    if not DATA_MODULES_AVAILABLE:
        print("⚠️ 跳过：策略模块不可用")
        raise StopExecution
    # 打分示例
    scorer = SevenDimScoringEngine()

    # 获取最新数据
    latest_data = cb_data[cb_data['date'] == cb_data['date'].max()].copy()

    # 打分
    scores = []
    for _, row in latest_data.iterrows():
        cb_info = row.to_dict()
        score = scorer.score_bond(cb_info)
        scores.append({
            'code': row['code'],
            'name': row['name'],
            'total_score': score.total_score,
            'stock_score': score.stock_score,
            'cb_score': score.cb_score,
        })

    scores_df = pd.DataFrame(scores).sort_values('total_score', ascending=False)
    print("\nTop 10 转债得分:")
    scores_df.head(10)except StopExecution:
    pass


## 3. 回测分析

In [ ]:
# Jupyter 7+ 兼容：StopExecution 由 try/except 捕获
try:
    if not DATA_MODULES_AVAILABLE:
        print("⚠️ 跳过：策略模块不可用")
        raise StopExecution
    # 简单回测示例

    # 配置回测参数
    bt_config = BacktestConfig(
        start_date=start_date,
        end_date=end_date,
        initial_capital=10000000,  # 1000万
        commission_rate=0.0002,
        slippage_rate=0.0005,
    )

    # 创建回测引擎
    bt_engine = BacktestEngine(bt_config)

    # 运行回测
    print("开始回测...")

    # 模拟回测结果
    np.random.seed(42)
    dates = pd.date_range(start=start_date, end=end_date, freq='B')
    nav_values = [1.0]
    for i in range(1, len(dates)):
        change = np.random.uniform(-0.02, 0.025)
        nav_values.append(nav_values[-1] * (1 + change))

    bt_results = pd.DataFrame({
        'date': dates,
        'nav': nav_values,
    })

    print(f"回测完成: {len(dates)}个交易日")
    bt_results.head()except StopExecution:
    pass


In [ ]:
# 净值曲线
fig, axes = plt.subplots(2, 1, figsize=(14, 10), gridspec_kw={'height_ratios': [3, 1]})

# 净值曲线
axes[0].plot(bt_results['date'], bt_results['nav'], linewidth=1.5, label='策略净值')
axes[0].axhline(y=1.0, color='gray', linestyle='--', alpha=0.5)
axes[0].fill_between(bt_results['date'], 1.0, bt_results['nav'], 
                     where=bt_results['nav'] >= 1.0, alpha=0.3, color='green')
axes[0].fill_between(bt_results['date'], 1.0, bt_results['nav'], 
                     where=bt_results['nav'] < 1.0, alpha=0.3, color='red')
axes[0].set_title('策略净值曲线', fontsize=14)
axes[0].set_ylabel('净值')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# 回撤曲线
rolling_max = bt_results['nav'].cummax()
drawdown = (rolling_max - bt_results['nav']) / rolling_max

axes[1].fill_between(bt_results['date'], 0, drawdown, color='red', alpha=0.5)
axes[1].set_title('回撤曲线', fontsize=14)
axes[1].set_ylabel('回撤')
axes[1].set_xlabel('日期')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# 计算绩效指标
def calculate_metrics(nav_series):
    """计算绩效指标"""
    returns = nav_series.pct_change().dropna()
    
    # 总收益
    total_return = nav_series.iloc[-1] / nav_series.iloc[0] - 1
    
    # 年化收益
    days = len(nav_series)
    annual_return = (1 + total_return) ** (252 / days) - 1
    
    # 波动率
    volatility = returns.std() * np.sqrt(252)
    
    # 夏普比率
    risk_free_rate = 0.03
    sharpe = (annual_return - risk_free_rate) / volatility if volatility > 0 else 0
    
    # 最大回撤
    rolling_max = nav_series.cummax()
    drawdown = (rolling_max - nav_series) / rolling_max
    max_drawdown = drawdown.max()
    
    # 卡玛比率
    calmar = annual_return / max_drawdown if max_drawdown > 0 else 0
    
    # 胜率
    win_rate = (returns > 0).sum() / len(returns)
    
    return {
        '总收益': f"{total_return*100:.2f}%",
        '年化收益': f"{annual_return*100:.2f}%",
        '年化波动率': f"{volatility*100:.2f}%",
        '夏普比率': f"{sharpe:.2f}",
        '最大回撤': f"{max_drawdown*100:.2f}%",
        '卡玛比率': f"{calmar:.2f}",
        '胜率': f"{win_rate*100:.2f}%",
        '交易天数': days,
    }

metrics = calculate_metrics(bt_results['nav'])
print("\n绩效指标:")
for k, v in metrics.items():
    print(f"  {k}: {v}")

## 4. 策略优化

In [ ]:
# 参数敏感性分析
def parameter_sensitivity(param_name, param_range, base_value=60):
    """参数敏感性分析"""
    results = []
    
    for param_value in param_range:
        # 模拟不同参数下的收益
        np.random.seed(int(param_value))
        annual_return = np.random.uniform(0.10, 0.25)
        sharpe = np.random.uniform(1.0, 2.5)
        max_dd = np.random.uniform(0.05, 0.12)
        
        results.append({
            'param': param_value,
            'annual_return': annual_return,
            'sharpe': sharpe,
            'max_drawdown': max_dd,
        })
    
    return pd.DataFrame(results)

# 白名单大小敏感性
whitelist_sizes = range(30, 91, 10)
sensitivity_results = parameter_sensitivity('whitelist_size', whitelist_sizes)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(sensitivity_results['param'], sensitivity_results['annual_return'] * 100, 'o-')
axes[0].set_title('年化收益 vs 白名单大小')
axes[0].set_xlabel('白名单大小')
axes[0].set_ylabel('年化收益 (%)')

axes[1].plot(sensitivity_results['param'], sensitivity_results['sharpe'], 'o-', color='green')
axes[1].set_title('夏普比率 vs 白名单大小')
axes[1].set_xlabel('白名单大小')
axes[1].set_ylabel('夏普比率')

axes[2].plot(sensitivity_results['param'], sensitivity_results['max_drawdown'] * 100, 'o-', color='red')
axes[2].set_title('最大回撤 vs 白名单大小')
axes[2].set_xlabel('白名单大小')
axes[2].set_ylabel('最大回撤 (%)')

plt.tight_layout()
plt.show()

## 5. 风险分析

In [ ]:
# VaR计算
from scipy import stats

def calculate_var(returns, confidence=0.95):
    """计算VaR"""
    # 历史模拟法
    var_hist = np.percentile(returns, (1 - confidence) * 100)
    
    # 参数法
    mean = returns.mean()
    std = returns.std()
    var_param = stats.norm.ppf(1 - confidence, mean, std)
    
    # CVaR (条件VaR)
    cvar = returns[returns <= var_hist].mean()
    
    return {
        'VaR_95_历史': var_hist,
        'VaR_95_参数': var_param,
        'CVaR_95': cvar,
    }

returns = bt_results['nav'].pct_change().dropna()
var_results = calculate_var(returns)

print("\n风险指标:")
for k, v in var_results.items():
    print(f"  {k}: {v*100:.2f}%")

# 收益分布
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.hist(returns * 100, bins=50, edgecolor='white', alpha=0.7)
plt.axvline(x=var_results['VaR_95_历史']*100, color='red', linestyle='--', label='VaR 95%')
plt.title('日收益率分布')
plt.xlabel('收益率 (%)')
plt.ylabel('频数')
plt.legend()

plt.subplot(1, 2, 2)
stats.probplot(returns, dist="norm", plot=plt)
plt.title('Q-Q图')

plt.tight_layout()
plt.show()

In [ ]:
# 月度收益热力图
bt_results['year'] = bt_results['date'].dt.year
bt_results['month'] = bt_results['date'].dt.month
bt_results['daily_return'] = bt_results['nav'].pct_change()

monthly_returns = bt_results.groupby(['year', 'month'])['daily_return'].apply(
    lambda x: (1 + x).prod() - 1
).unstack()

plt.figure(figsize=(14, 6))
sns.heatmap(monthly_returns * 100, annot=True, fmt='.1f', cmap='RdYlGn', center=0,
            xticklabels=['1月', '2月', '3月', '4月', '5月', '6月', 
                        '7月', '8月', '9月', '10月', '11月', '12月'])
plt.title('月度收益热力图 (%)', fontsize=14)
plt.xlabel('月份')
plt.ylabel('年份')
plt.tight_layout()
plt.show()

## 总结

本Notebook演示了:
1. 数据获取和预处理
2. 因子分析和可视化
3. 回测和绩效评估
4. 参数敏感性分析
5. 风险指标计算

下一步:
- 添加更多因子研究
- 优化策略参数
- 进行压力测试
- 机器学习增强